# Конспект. Модуль 2: Ансамбли — Bagging и Random Forest

## 1. Зачем это нужно и как это связано с Модулем 1

В конце прошлого модуля мы пришли к ключевому выводу: **одиночное глубокое дерево — модель с почти нулевым bias и огромной variance**. Оно идеально подстраивается под обучающую выборку, но крайне нестабильно — малое изменение данных полностью меняет структуру дерева.

Возникает естественный вопрос: если у нас есть модель с низким bias, но высокой variance, можем ли мы **погасить variance, не трогая bias**? Ответ — да, если мы построим **много** таких нестабильных моделей на слегка разных данных и **усредним** их предсказания. Это и есть идея **Bagging** (Bootstrap Aggregating), а **Random Forest** — её усиленная версия специально для деревьев.

Важно сразу отметить принципиальное отличие от того, что будет в Модулях 3+: в bagging все модели строятся **независимо и параллельно** — они не «знают» друг о друге и не пытаются исправить ошибки друг друга. Это прямая противоположность **бустингу**, где каждая следующая модель строится **последовательно**, специально чтобы исправить ошибки предыдущих. Здесь заложена главная развилка курса — держите её в голове.

## 2. Bootstrap: математика «пересэмплирования»

**Bootstrap** — это сэмплирование **с возвращением** (with replacement) из обучающей выборки размера `N`, при котором формируется новая выборка того же размера `N`, где некоторые исходные объекты могут повторяться, а некоторые — не попасть вовсе.

**Пример на 8 объектах** (используем тот же игрушечный датасет транзакций из Модуля 1, пронумерованный 1–8). Один bootstrap-сэмпл может выглядеть так:

In [ ]:
Исходная выборка:     [1, 2, 3, 4, 5, 6, 7, 8]
Bootstrap-сэмпл #1:   [3, 3, 1, 8, 5, 5, 2, 6]   <- объект 3 и 5 повторились, объекты 4 и 7 не попали

**Сколько в среднем объектов НЕ попадёт в bootstrap-сэмпл?** Это классический и полезный расчёт, который стоит уметь вывести самостоятельно, тем более что у вас сильная база в математике.

Вероятность, что **конкретный** объект не будет выбран **на одном** из `N` розыгрышей: `(1 - 1/N)`.
Вероятность, что он не будет выбран **ни разу за N** независимых розыгрышей (это и есть размер bootstrap-сэмпла): `(1 - 1/N)^N`.

Возьмём предел при `N -> ∞` — это классический предел из математического анализа:

In [ ]:
lim(N->∞) (1 - 1/N)^N = e^(-1) ≈ 0.368

**Вывод:** в среднем **~36.8%** объектов исходной выборки не попадают в конкретный bootstrap-сэмпл (даже при `N=8` это уже приближение `(7/8)^8 ≈ 0.344`, довольно близко к асимптотике). Значит, **~63.2%** объектов попадают — некоторые по одному разу, некоторые несколько раз.

Этот факт — не просто математическое любопытство. Именно эти «непопавшие» ~36.8% объектов и станут основой для **Out-of-Bag оценки** в разделе 5 — по сути, бесплатной валидации без отдельного holdout.

## 3. Bagging: алгоритм и почему он снижает именно variance, а не bias

**Алгоритм Bagging (общий, не только для деревьев):**

1. Сформировать `B` bootstrap-выборок из обучающих данных (обычно `B` = число деревьев, `n_estimators`).
2. Обучить **одинаковый по типу** базовый алгоритм (в нашем случае — дерево решений, желательно **глубокое, непостриженное**) на каждой bootstrap-выборке независимо.
3. Для предсказания нового объекта:
   - **Классификация:** голосование большинством (majority vote) среди всех `B` моделей, либо усреднение вероятностей (`predict_proba`, что обычно даёт лучший результат, чем жёсткое голосование).
   - **Регрессия:** простое среднее арифметическое предсказаний всех `B` моделей.

### 3.1. Формальный вывод, почему усреднение снижает variance

Пусть у нас `B` моделей, каждая с одинаковой variance `σ²` (все построены одним и тем же алгоритмом на bootstrap-выборках из одного распределения). Если бы модели были **полностью независимы** (что было бы верно, если бы каждая обучалась на совершенно отдельной, независимо собранной выборке), variance усреднённого предсказания была бы:

In [ ]:
Var(среднее) = σ² / B

То есть при `B=100` независимых моделях variance снижается в 100 раз — потрясающий результат. **Но модели bagging не независимы** — все bootstrap-выборки взяты из **одной и той же** исходной выборки, поэтому предсказания моделей **коррелированы** между собой (коэффициент корреляции `ρ`). Точная формула для variance среднего `B` **коррелированных** одинаково распределённых величин:

In [ ]:
Var(среднее) = ρ·σ² + (1-ρ)·σ²/B

**Разбор формулы:** при `B -> ∞` второе слагаемое стремится к нулю, но первое слагаемое `ρ·σ²` **остаётся** — это «пол», ниже которого variance не опустится, сколько бы деревьев мы ни добавляли. **Чем меньше `ρ` (корреляция между деревьями), тем ниже этот пол и тем эффективнее bagging.**

**Это и есть главная мотивация Random Forest** — если бы мы просто делали bagging над обычными деревьями (каждое дерево видит все признаки), деревья были бы довольно сильно коррелированы: если в данных есть один очень сильный признак, **почти каждое** bootstrap-дерево выберет именно его для первого разбиения — деревья получаются структурно похожими, `ρ` высокий. Random Forest добавляет **второй источник случайности** специально, чтобы снизить `ρ`.

## 4. Random Forest = Bagging + случайное подпространство признаков

Random Forest модифицирует шаг 2 алгоритма bagging: при поиске лучшего разбиения **в каждом узле** дерево рассматривает **не все `M` признаков**, а случайное подмножество из `m < M` признаков (выбирается заново в каждом узле).

**Типичные значения `m` (гиперпараметр `max_features`):**
- Классификация: `m = sqrt(M)` — например, при 100 признаках рассматривается ~10 в каждом узле.
- Регрессия: `m = M/3`.

**Почему это работает:** даже если в данных есть один доминирующий признак, он не всегда попадёт в случайную подвыборку `m` признаков конкретного узла — иногда дереву **придётся** искать разбиение среди других, менее очевидных признаков. Это заставляет разные деревья ансамбля исследовать разные структуры зависимостей, снижая `ρ` из формулы выше — прямое, измеримое снижение variance ансамбля.

**Важный побочный эффект:** снижение `max_features` немного **увеличивает bias** каждого отдельного дерева (оно лишено доступа к лучшему признаку в моменте разбиения), но за счёт декорреляции суммарный **variance ансамбля** падает сильнее, чем растёт bias — итоговый trade-off чаще всего в плюсе. Это ещё один практический пример bias-variance компромисса, но уже на уровне ансамбля, а не одного дерева.

**Итого, в Random Forest два независимых источника случайности:**
1. Bootstrap-сэмплирование **объектов** (строк) — это уже bagging.
2. Случайное подмножество **признаков** в каждом узле — это добавка Random Forest.

Отсюда и название: лес из деревьев, каждое из которых видит и свою случайную подвыборку строк, и свою случайную подвыборку признаков в каждом разбиении.

## 5. Out-of-Bag (OOB) error — бесплатная валидация

Из раздела 2 мы знаем: для каждого дерева примерно 36.8% объектов **не попали** в его bootstrap-обучающую выборку — это **out-of-bag** объекты для данного конкретного дерева.

**Идея OOB-оценки:** для каждого объекта `x_i` обучающей выборки найдём **только те деревья**, для которых этот объект был out-of-bag (не участвовал в обучении), и усредним (или проголосуем по большинству) их предсказания для `x_i`. Получаем `OOB-предсказание` для каждого обучающего объекта — фактически, предсказание модели, которая **не видела** этот объект во время обучения, хотя формально мы не выделяли отдельный holdout.

**Формально:**

In [ ]:
OOB_error = (1/N) · Σ(i=1..N) Loss( y_i, agg{ tree_b(x_i) : x_i ∉ bootstrap_b } )

где `agg` — усреднение/голосование только по деревьям `b`, для которых `x_i` был out-of-bag.

**Почему это ценно:**
- Не нужно **жертвовать** частью данных на отдельный validation split — весь датасет идёт в обучение, но при этом получаем честную оценку обобщающей способности.
- В sklearn: `RandomForestClassifier(oob_score=True)`, после `.fit()` доступен атрибут `.oob_score_`.
- OOB-оценка асимптотически близка к оценке через K-Fold кросс-валидацию (тот же порядок величины ошибки), но вычисляется **за один проход обучения**, а не за K отдельных обучений — существенная экономия времени на больших данных.

**Важная оговорка:** OOB работает только потому, что bagging строит деревья **независимо** и у каждого дерева есть свой персональный набор «невиданных» объектов. В бустинге (Модули 3+) такой трюк **невозможен** — там каждое следующее дерево строится на основе ошибок **всех предыдущих**, поэтому все данные последовательно проходят через весь ансамбль, и понятия «out-of-bag для дерева» просто не существует. Это ещё одна практическая иллюстрация разницы параллельных и последовательных ансамблей.

## 6. Численный пример: как работает голосование и усреднение

Возьмём 5 деревьев (`B=5`), обученных на разных bootstrap-выборках нашего игрушечного антифрод-датасета, и посмотрим, как они предсказывают для нового объекта `x_new`:

| Дерево | Предсказание (класс) | Вероятность fraud (`predict_proba`) |
|---|---|---|
| Дерево 1 | 1 (fraud) | 0.82 |
| Дерево 2 | 0 (честная) | 0.35 |
| Дерево 3 | 1 (fraud) | 0.61 |
| Дерево 4 | 1 (fraud) | 0.55 |
| Дерево 5 | 0 (честная) | 0.40 |

**Жёсткое голосование (majority vote):** 3 дерева за класс 1, 2 — за класс 0 -> итоговое предсказание = **1 (fraud)**.

**Усреднение вероятностей (мягкое голосование, `predict_proba`):**

In [ ]:
p_fraud = (0.82 + 0.35 + 0.61 + 0.55 + 0.40) / 5 = 2.73 / 5 = 0.546

Итоговая вероятность **0.546** — тоже выше порога 0.5, значит класс 1, но обратите внимание: усреднение вероятностей несёт **больше информации**, чем жёсткое голосование (мы видим, что уверенность модели умеренная, 0.546, а не «уверенное большинство 3 из 5»). На практике `predict_proba`-усреднение почти всегда работает лучше жёсткого голосования, особенно если вам важен не только класс, но и калиброванная оценка риска (как в антифроде) — поэтому в sklearn `RandomForestClassifier.predict()` под капотом на самом деле использует именно усреднение вероятностей, а не жёсткое голосование.

## 7. Гиперпараметры Random Forest

| Параметр | Что делает | Практический совет |
|---|---|---|
| `n_estimators` | Число деревьев в лесу | Больше — почти всегда лучше (variance продолжает падать по формуле из раздела 3.1), но с диминishing returns; ограничение — время обучения/инференса. Обычно 100–1000 |
| `max_features` | Сколько признаков рассматривать в узле | Главный рычаг декорреляции деревьев; `sqrt` для классификации, `1/3` для регрессии — хорошие дефолты, стоит тюнить в узком диапазоне |
| `max_depth` | Глубина каждого дерева | Обычно **не ограничивают** (или ставят большое значение) — в отличие от одиночного дерева, здесь глубокие/переобученные деревья — это нормально и даже желательно, variance гасится усреднением |
| `min_samples_leaf` | Мин. объектов в листе | В отличие от `max_depth`, этот параметр иногда полезно немного увеличить (2-5) для сглаживания шумных данных |
| `bootstrap` | Использовать ли bootstrap-сэмплирование строк | `True` по умолчанию — это и есть «bagging»-часть; `False` превращает алгоритм в «Extra Trees»-подобный вариант без пересэмплирования строк |
| `max_samples` | Доля/число объектов в каждом bootstrap-сэмпле | По умолчанию — размер как у исходной выборки (с повторами); можно уменьшить для ускорения на очень больших данных |
| `oob_score` | Считать ли OOB-ошибку | `True` — бесплатная оценка качества без отдельного holdout (раздел 5) |
| `n_jobs` | Число потоков для параллельного обучения деревьев | Так как деревья независимы, можно смело ставить `-1` (все ядра) — это **невозможно** в такой же степени для бустинга, где деревья строятся последовательно |
| `criterion` | Критерий разбиения узла | Тот же смысл, что и в Модуле 1 (`gini`/`entropy`/`squared_error` и т.д.) |

**Важное наблюдение:** обратите внимание на асимметрию с Модулем 1 — там `max_depth` и `min_samples_leaf` были **главными** параметрами против переобучения одного дерева. Здесь же деревья специально оставляют глубокими, а против переобучения ансамбля работает в первую очередь `max_features` (декорреляция) и `n_estimators` (усреднение). Это прямое следствие разной роли отдельного дерева в одиночной модели и в ансамбле.

## 8. Практика: код

### 8.1. Random Forest vs одиночное дерево

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import roc_auc_score

X, y = make_classification(n_samples=2000, n_features=20,
                            n_informative=10, weights=[0.9, 0.1],
                            random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Одиночное дерево без ограничений — переобучится
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
print("Single tree — train ROC-AUC:", roc_auc_score(y_train, tree.predict_proba(X_train)[:, 1]))
print("Single tree — test ROC-AUC:", roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1]))

# Random Forest с теми же (неограниченными) деревьями внутри
rf = RandomForestClassifier(n_estimators=300, max_features="sqrt",
                             oob_score=True, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
print("Random Forest — train ROC-AUC:", roc_auc_score(y_train, rf.predict_proba(X_train)[:, 1]))
print("Random Forest — test ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]))
print("Random Forest — OOB score:", rf.oob_score_)

**Что ожидать:** одиночное дерево покажет почти идеальный train ROC-AUC (~1.0) и заметно более слабый test — классический разрыв переобучения из Модуля 1. Random Forest даст train ROC-AUC чуть ниже идеала (усреднение всё же немного сглаживает даже train-предсказания), но **test ROC-AUC заметно выше**, чем у одного дерева — и близко к `oob_score_`, что подтверждает: OOB-оценка действительно хорошо аппроксимирует качество на невиданных данных.

### 8.2. Эффект `n_estimators` — variance убывает, но с насыщением

In [ ]:
import matplotlib.pyplot as plt

n_trees_range = [1, 5, 10, 25, 50, 100, 200, 400]
test_scores = []

for n in n_trees_range:
    rf = RandomForestClassifier(n_estimators=n, max_features="sqrt",
                                 n_jobs=-1, random_state=42)
    rf.fit(X_train, y_train)
    test_scores.append(roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]))

plt.plot(n_trees_range, test_scores, marker="o")
plt.xlabel("n_estimators")
plt.ylabel("Test ROC-AUC")
plt.title("Качество растёт и выходит на плато — в отличие от переобучения по глубине в Модуле 1")
plt.show()

**Ключевое отличие от графика в Модуле 1:** там увеличение `max_depth` в какой-то момент **ухудшало** test-метрику (переобучение). Здесь увеличение `n_estimators` **никогда не ухудшает** ожидаемое качество (максимум — выходит на плато и просто тратит время впустую) — это прямое следствие формулы `Var(среднее) = ρσ² + (1-ρ)σ²/B`: добавление деревьев может только уменьшать или не менять variance, но не увеличивать её.

### 8.3. Эффект `max_features` — декорреляция в действии

In [ ]:
for mf in [1, 3, 5, "sqrt", 15, 20]:  # 20 - все признаки = обычный bagging без доп. декорреляции
    rf = RandomForestClassifier(n_estimators=200, max_features=mf,
                                 n_jobs=-1, random_state=42)
    rf.fit(X_train, y_train)
    score = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
    print(f"max_features={mf}: test ROC-AUC={score:.4f}")

При `max_features=20` (все признаки) вы фактически получаете обычный bagging **без** дополнительной декорреляции Random Forest — сравните этот результат с `max_features="sqrt"`, чтобы увидеть эффект декорреляции своими глазами на конкретных цифрах.

## 9. Bias-Variance: сводная картина трёх моделей

| Модель | Bias | Variance | Почему |
|---|---|---|---|
| Одно глубокое дерево | Очень низкий | Очень высокий | Идеально подстраивается под train, крайне чувствительно к изменению данных (Модуль 1) |
| Random Forest (много глубоких деревьев) | Почти как у одного дерева (немного выше из-за `max_features`) | Сильно снижен усреднением и декорреляцией | Усреднение коррелированных высоко-дисперсных оценок гасит variance, оставляя bias почти нетронутым |
| (Забегая вперёд) Градиентный бустинг | Снижается последовательно, шаг за шагом | Контролируется отдельно (learning rate, ранняя остановка) | Принципиально другой механизм — не усреднение, а последовательная коррекция ошибок (Модуль 3+) |

Это ключевая сводная таблица курса: Random Forest **атакует variance**, оставляя bias примерно на уровне одного (пусть и переобученного) дерева. Если данные требуют модели с **более низким bias**, чем может дать бэггинг глубоких деревьев (например, сложные нелинейные взаимодействия, которые ни одно отдельное дерево не улавливает полностью) — бэггинг здесь упирается в потолок. Именно эту стену и обходит бустинг совершенно другим механизмом — это будет подробно в Модуле 3.

## 10. Частые вопросы на собеседовании

| Вопрос | На что обратить внимание в ответе |
|---|---|
| Почему в Random Forest используют глубокие, а не постриженные деревья? | Bagging специально нацелен на снижение variance через усреднение; для этого нужны базовые модели с низким bias (глубокие деревья), variance гасится ансамблированием |
| Зачем нужен `max_features`, если уже есть bootstrap по строкам? | Bootstrap снижает корреляцию между деревьями лишь частично — при наличии доминирующего признака деревья всё равно похожи структурно; `max_features` декоррелирует их сильнее, что напрямую снижает variance ансамбля по формуле `ρσ² + (1-ρ)σ²/B` |
| Чем OOB-ошибка отличается от K-Fold CV? | Похожий смысл (оценка на «невиданных» данных), но OOB вычисляется за один проход обучения ансамбля, а не за K отдельных обучений — быстрее, хотя асимптотически даёт похожую оценку |
| Может ли Random Forest переобучиться при увеличении `n_estimators`? | Практически нет — variance ансамбля не растёт с числом деревьев (только убывает или выходит на плато); переобучение Random Forest регулируется в первую очередь глубиной/сложностью отдельных деревьев и `max_features`, а не их числом |
| В чём принципиальное отличие Bagging от Boosting на уровне идеи? | Bagging — параллельные независимые модели + усреднение -> атака на variance. Boosting — последовательные зависимые модели, каждая исправляет ошибки предыдущей -> атака на bias |

## 11. Чек-поинт — попробуйте ответить без подсказок

1. Почему Random Forest почти не переобучается при увеличении числа деревьев, а одиночное дерево — переобучается при увеличении глубины?
2. В чём разница между Bagging и Random Forest конкретно — какие два источника случайности задействует Random Forest и зачем нужен именно второй?
3. Выведите (или объясните словами), почему доля объектов, не попадающих в конкретный bootstrap-сэмпл, стремится к `e^(-1) ≈ 36.8%`.
4. Почему OOB-оценка невозможна в градиентном бустинге, но возможна в bagging/Random Forest?
5. Если бы деревья в ансамбле были **полностью некоррелированы** (`ρ=0`), что произошло бы с variance ансамбля при `B -> ∞`? А если бы деревья были **полностью идентичны** (`ρ=1`)?

## Ответы для самопроверки

<details>
<summary>Раскрыть после того, как попробуете ответить сами</summary>

1. Одиночное дерево при росте глубины снижает bias, но неограниченно наращивает variance — переобучение прямое следствие. Random Forest при росте `n_estimators` **не меняет** структуру/сложность отдельных деревьев (их глубина фиксирована гиперпараметрами), а лишь усредняет всё больше независимых оценок — по формуле `Var(среднее) = ρσ² + (1-ρ)σ²/B` variance ансамбля может только убывать или выходить на плато, никогда не расти от увеличения `B`.

2. Первый источник — bootstrap-сэмплирование **объектов** (строк) для каждого дерева, это и есть базовый bagging. Второй, добавленный именно в Random Forest — случайное подмножество **признаков**, рассматриваемых при поиске лучшего разбиения в каждом узле каждого дерева. Второй источник нужен, чтобы дополнительно **декоррелировать** деревья: без него, при наличии сильного доминирующего признака, почти все bootstrap-деревья всё равно выбирали бы его первым и получались структурно похожими (высокий `ρ`), что ограничивает выигрыш от усреднения по формуле variance.

3. Вероятность, что конкретный объект не будет выбран на одном розыгрыше из `N` — `(1 - 1/N)`. За `N` независимых розыгрышей (ровно столько, сколько объектов в bootstrap-сэмпле того же размера) вероятность ни разу не попасть — `(1-1/N)^N`. Это классическое выражение, предел которого при `N->∞` по определению числа `e` равен `e^(-1) ≈ 0.368`. Даже при небольших `N` (например, 8) значение уже близко к этому пределу.

4. OOB работает потому, что каждое дерево в bagging/Random Forest обучается **независимо** на своей bootstrap-выборке, и у каждого дерева есть персональный набор «невиданных» (~36.8%) объектов — можно собрать честную оценку из предсказаний деревьев-«чужаков» для каждого объекта. В градиентном бустинге деревья строятся **последовательно**: каждое следующее дерево обучается на псевдо-остатках, вычисленных с использованием предсказаний **всех предыдущих** деревьев на **всех** объектах — там нет отдельного, независимого от остального ансамбля «взгляда» на данные, поэтому концепция OOB не применима в том же виде.

5. При `ρ=0` (полностью независимые деревья): `Var(среднее) = 0·σ² + (1-0)·σ²/B = σ²/B -> 0` при `B->∞` — variance ансамбля стремится к нулю, идеальный случай. При `ρ=1` (все деревья идентичны, например, если убрать всю случайность): `Var(среднее) = 1·σ² + 0·σ²/B = σ²` — усреднение **вообще не помогает**, variance ансамбля равна variance одного дерева, сколько бы деревьев мы ни добавляли. Это объясняет, почему борьба за снижение `ρ` (через bootstrap и `max_features`) — не второстепенная деталь, а суть того, почему Random Forest вообще работает.

</details>